# Data Science in Psychology and Neuroscience

## Class info:
* Week #12
* Day: April 7, 2026
* Time: 9:30—10:45 AM
* Location: Logan Hall 125
* <a href="https://forms.microsoft.com/r/26vAcJWrwH">Click here to submit your attendance for Week 12, Question 21!</a>
  
## Instructor info:
* Dr. Jeremy Hogeveen
* jhogeveen@unm.edu
* Logan Hall 281 (Office Hours By Appointment)
  
## Syllabus:
* <a href="https://www.dropbox.com/scl/fi/6fs6fi4kvkwtxn7j7x8ua/PSY450650_DSPN_Spring2026_Syllabus.pdf?rlkey=148e5t4ah8q2n1daclt7mp0h6&dl=0">Download here</a>

## Today's topic:
* Modeling our ketamine trial data, continued
1. Linear Mixed Models (LMMs).
2. Testing the assumptions of the GLM.
3. My suggestions for GLM violations: Use robust estimators.

 

# Section 1. Linear Mixed Models

## 1.1 Refresher: We started by running a `3 (time: pre, posttest, followup)` x `3 (drug: ketamine, amphetamine, placebo)` Mixed ANOVA (omnibus test)
*  Are there differences in the long-term antidepressant effects of ketamine versus amphetamine versus placebo?
*  This represents the omnibus test for this design.
    *  AKA, what we'd actually run first if this was our study. Why?
        * Multiple comparisons issues:
            * At this point, we'd run i) a `paired t-test`, ii) an `independent t-test`, iii) a `one-way ANOVA`, and iv) a `rm-ANOVA`.
            * Family-Wise Error Rate (FWER): $FWER =  1 - (1 - \alpha)^n$.
                * $FWER = 1 - (1 - 0.95)^4$
                * $FWER = 0.1855$
                * False positive rate has ≈quadrupled.
        * Maximizes statistical power:
            * Using all subjects + data points to estimate error variance.
            * Higher error degrees of freedom in the ANOVA denominator, lower critical threshold for significance.
        * Capturing the critical `interaction term` (drug * time)
            * Does the trajectory of change depend on the drug? Only the interaction term in the omnibus model does that.

In [12]:
# package loadings
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(rstatix))
suppressPackageStartupMessages(library(afex))
suppressPackageStartupMessages(library(emmeans))
suppressPackageStartupMessages(library(knitr))
suppressPackageStartupMessages(library(lme4))
suppressPackageStartupMessages(library(lmerTest))
suppressPackageStartupMessages(library(ggeffects))
suppressPackageStartupMessages(library(gridExtra))

In [13]:
# Load in our data
df <- read_csv('../data/synth_ketamine_data.csv')

# Reorder our factors accordingly
Time_desired_order <- c("Pre", "Post", "Followup")
df$Time <- factor(df$Time, levels = Time_desired_order)
Drug_desired_order <- c("Placebo", "Amphetamine", "Ketamine")
df$Drug <- factor(df$Drug, levels = Drug_desired_order)

head(df)

Rows: 180 Columns: 5
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (2): Drug, Time
dbl (3): Subject, Change, HAMD

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Subject,Drug,Change,Time,HAMD
<dbl>,<fct>,<dbl>,<fct>,<dbl>
1,Ketamine,10.89830,Pre,28.112875
1,Ketamine,10.89830,Post,17.214579
1,Ketamine,10.89830,Followup,15.227329
2,Ketamine,12.55569,Pre,22.305905
2,Ketamine,12.55569,Post,9.750214
2,Ketamine,12.55569,Followup,7.809342


In [ ]:
# visualize the effects
df_mixed_anova <- df %>%
    group_by(Drug, Time) %>%
    get_summary_stats(HAMD, type = "mean_sd")
# print(df_mixed_anova)

# plotting the effects
dodge <- position_jitterdodge(jitter.width = 0.2, dodge.width = 0.6)
p_depression_over_time <- ggplot(df, aes(x = Time, y = HAMD, fill = Drug, color=Drug)) +
  geom_jitter(shape= 21, color="black", size = 2, alpha = 0.3, position = dodge) +
  stat_summary(fun.data = "mean_se", geom = "errorbar", width = 0.2, position = dodge) +
  stat_summary(fun = "mean", geom = "point", size = 4, position = dodge) +
  stat_summary(aes(group = Drug),fun = "mean", geom = "line", size = 1, position = dodge) +
  theme_classic(base_size = 16) +
  ggtitle("HAM-D Scores by Treatment and Time") +
  xlab("Timepoint") +
  ylab("HAM-D Score")
plot(p_depression_over_time)

In [ ]:
# Running the mixed mode
m_mixed_anova <- aov_ez(
    id = "Subject",
    dv = "HAMD",
    data = df,
    between = "Drug",
    within = "Time"
    )
summary(m_mixed_anova)

# running the EMmeans function
means_mixed <- emmeans(m_mixed_anova, pairwise ~ Drug | Time, adjust = "tukey")
kable(means_mixed$emmeans,digits=3, caption = "EMMeans from mixed ANOVA")
kable(means_mixed$contrasts,digits=3, caption = "PostHoc Tests from mixed ANOVA")

In [ ]:
# Visualizing the effects
df_mixed_anova_summary <- df %>%
  group_by(Drug, Time) %>%
  summarise(mean_hamd = mean(HAMD),
            se_hamd = sd(HAMD) / sqrt(n()))

# Plotting the effects
p_mixed_anova <- ggplot(df_mixed_anova_summary, aes(x = Time, y = mean_hamd, color = Drug, group = Drug)) +
  geom_line(size = 1.2) +
  geom_point(size = 4) +
  geom_errorbar(aes(ymin = mean_hamd - se_hamd, ymax = mean_hamd + se_hamd), width = 0.1) +
  theme_classic(base_size=16)
p_mixed_anova

In [ ]:
# Running the mixed model
m_mixed_anova <- aov_ez(id = "Subject", dv = "HAMD", between = "Drug", within = "Time", data = df)
summary(m_mixed_anova)

# running the estimated marginal means
means_mixed <- emmeans(m_mixed_anova, pairwise ~ Drug | Time, adjust = "tukey")
kable(means_mixed$emmeans, digits = 3, caption = "EMmeans from Mixed 3x3 ANOVA")
kable(means_mixed$contrasts, digits = 3, caption = "Post-Hoc Tests from Mixed 3x3 ANOVA")

### 1.2 Key scenarios when linear mixed models (LMMs) are superior to ANOVA.
* Missing data.
* Unbalanced designs.
* Autocorrelated data.
* Timeseries data.
* Longitudinal data.
* Complex nested data structures.
    * e.g. Students within classrooms, patients within diagnostic categories, animals within a litter, etc.
* Continuous time.
* Strong violations of sphericity assumptions in rmANOVA.

### 1.3 Potential solution: Linear mixed models (LMMs)
* AKA mixed models, multilevel models, hierarchical models.
* LMMs don't rely on perfect, balanced matrices.
    * They look at all available data points.
    * If a subject has a Pre and Post score but no Followup, the LMM still uses their Pre and Post data to estimate the overall trajectories!
* Model works by maximizing the likelihood of the model given all the data we have.

### 1.4 How they work
* To understand how Linear Mixed Models handle this data, we need to split our variables into two distinct categories: __Fixed Effects__ and __Random Effects__.
    * Fixed Effects (`Drug` and `Time`): These are the experimental variables we actually care about testing and generalizing to the broader population. We want to know the universal, average effect of Ketamine versus Placebo over time.
    * Random Effects (`Subject`): This accounts for the idiosyncratic "noise" in our specific sample.
* The `R` Syntax: $HAMD ~ Drug * Time + (1 | Subject)$
    * `Drug` * `Time` represents our __Fixed Effects__.
    * `(1 | Subject)` tells the model to add a __Random Intercept__ for each person.
* In plain English: "Calculate the overall, average effects of the Drug over Time, but allow every single subject to start at their own unique baseline level of depression."

### 1.5 Visualizing the LMM interaction estimated marginal means

* Notice that the EMMs from the model map pretty directly onto the raw data plot
    * What happens if we covary for age and sex?

In [ ]:
# adding age, between 20 and 60, to the data
df <- df %>%
  group_by(Subject) %>%
  mutate(
    Age = pmin(pmax(round(rnorm(1, mean = 35, sd = 5)), 20), 60),
    Sex = sample(c("M", "F"), 1)) %>%
  ungroup() %>%
  mutate(Sex = fct_relevel(as.factor(Sex), "M"))
hist(df$Age)
table(df$Sex, exclude = NULL)

### 1.6 The flexibility of LMMs
* LMMs can handle designs much more sophisticated than our simple `(1 | Subject)` random intercept.
* Two incredibly common extensions in clinical and neuroscience research are:
    * Nested Designs:
        * What if our sample included patients with two different underlying conditions (e.g., Major Depressive Disorder vs. Bipolar Depression)?
        * We can model this explicitly as a ___nested random intercept___
            * Changing our syntax to `(1 | Diagnosis/Subject)` tells the model: _"Subjects belong to specific Diagnostic groups; calculate an overarching baseline for MDD vs. BPD, and then calculate each individual Subject's baseline within their respective group."_
    * Random Slopes:
      * Our model assumes the drug worked at the exact same _rate_ for everyone (i.e., random intercepts, parallel slopes over Time).
      * In reality, Ketamine might drop depression scores incredibly fast for Subject 1, but very slowly for Subject 2, etc. 
      * We can add a ___Random Slope___ to explicitly model this variance
          * Changing our syntax to `(1 + Time | Subject)` tells the model: _"Give everyone their own starting baseline (intercept), AND allow everyone to have their own unique trajectory or rate of change (slope) over time."_
            
<img src="img/LMMs.png" width=600>
(<a href="https://peerj.com/articles/4794/">figure ref</a>)

# Section 2. General Linear Model (GLM) Assumptions.

<img src="img/decision_tree.png" width=300>

0. Outliers & influential observations
1. Independence
2. Normality
3. Equal variances

## 2.1 Before discussing each in turn, let's "break" our data a bit.

In [ ]:
df_bad <- df %>%
  mutate(
    # Injecting conditional Skewness (a noticeable right-skew and heteroscedasticity for the assumptions check)
    HAMD = ifelse(Drug == "Ketamine" & Time != "Pre", 
                  HAMD + rexp(n(), rate = 0.4), 
                  HAMD),
    
    # Adding 15 points to just two random subjects.
    HAMD = ifelse(Subject %in% c(5, 12) & Time == "Post", 
                  HAMD + 15, 
                  HAMD)
  )

### 2.2 Hunting for influential observations and outliers

* Outliers:

* Influential Observations:

### 2.3 GLM Assumptions: 


#### Independence
* ___Observations are independent of each other___
* Repeated-measures designs
    * e.g. within-subject observations across time often show "serial correlation" or "autocorrelation"
* Nested designs (e.g. workers on a team, team within a department, departments within an org.)
* For `lm`, use `durbinWatsonTest(model)`
* For `lmer`, check the residuals plot

#### Normality
* ___There is an assumption that the data are normally distributed___
* Most datasets contain at least one non-gaussian variable...
    * That doesn't _necessarily_ matter. The key is: Are the model error terms (i.e., residuals) normally distributed?
* For `lm`, use `shapiro.test(model)`
* For `lmer`, check the residuals plot

#### Equal variances 
* ___Across different types of GLM-family tests, there are common assumptions about equal variances___
* Homogeneity of variance: Differences, single predictor variable (e.g. t.test, one-way ANOVA)
* Sphericity: Differences, multilevel factor(s) (e.g. repeated-measures and mixed model ANOVA)
* Homoscedasticity: Continuous, do predictor variables have similar variance (e.g. multiple regression)
* For ...
    * `t.test`, use `var.test`
    * `anova`, use `bartlett.test`
    * `lm`, use `car::ncvTest`
* For `lmer`, check the residuals plot